# RG3 전체 가스19개 탐지 여부 점검

같은 .02 내부 정상 분위수 임계값 설정을 사용한 82~85 예측 연결. score-threshold 부호는 왜 판정에서 놓쳤는지 보여주지만 물리적 가스 원인 설명이 아니다. 전체 가스 라벨을 보는 개발용 사후 분석이며 이를 통한 이후 피처 선택은 독립 검증이 아님.


In [1]:
from pathlib import Path
import pandas as pd,numpy as np
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
files=[('std4_reference','rg3_absolute_state',.02),('absolute_state10','rg3_absolute_state',.02),('crossfit_residual8','rg3_crossfit_residual',.02),('causal_local_relation8','rg3_causal_local_relation',.02)]
result=None
for model,prefix,budget in files:
 p=pd.read_csv(t/(prefix+'_predictions.csv'))
 g=p[(p.model==model)&(p.budget==budget)&(p.actual==1)][['TimeStamp','block','actual','prediction','score','threshold']].copy()
 assert len(g)==19 and not g.TimeStamp.duplicated().any()
 g['threshold_margin']=g.score-g.threshold
 g=g.rename(columns={c:model+'__'+c for c in ['prediction','score','threshold','threshold_margin']})
 if result is None:result=g
 else:result=result.merge(g.drop(columns=['block','actual']),on='TimeStamp',validate='one_to_one')
 f=pd.read_csv(t/'rg3_causal_local_relation_features.csv')
 if model==files[-1][0]:result=result.merge(f,on='TimeStamp',validate='one_to_one')
result['state_new_vs_std']=(result.absolute_state10__prediction==1)&(result.std4_reference__prediction==0)
result.to_csv(t/'rg3_relationship_all19_gas_review.csv',index=False,encoding='utf-8-sig')
print(result[['TimeStamp','block','std4_reference__prediction','absolute_state10__prediction','crossfit_residual8__prediction','causal_local_relation8__prediction']].to_string(index=False))
print('New gas versus matched std4 baseline:',int(result.state_new_vs_std.sum()))


          TimeStamp  block  std4_reference__prediction  absolute_state10__prediction  crossfit_residual8__prediction  causal_local_relation8__prediction
2020-10-22 00:51:52      2                           0                             0                               0                                   0
2020-10-22 01:04:15      2                           0                             0                               0                                   0
2020-10-22 01:26:54      2                           0                             0                               0                                   0
2020-10-22 01:57:47      2                           0                             0                               0                                   0
2020-10-22 04:46:36      3                           0                             0                               0                                   0
2020-10-22 04:51:46      3                           0                            